# Fixed and Tuned SVM Model for Online Purchase Prediction

## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load and Clean Dataset

In [ ]:

df = pd.read_excel("online_purchase_prediction_100000_with_duplicates.xlsx")
df.replace("NaN", np.nan, inplace=True)

print("Initial shape:", df.shape)

df.drop_duplicates(inplace=True)

if "UserID" in df.columns:
    df.drop("UserID", axis=1, inplace=True)

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include="object").columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

df[num_cols] = num_imputer.fit_transform(df[num_cols])

if len(cat_cols) > 0:
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("Shape after cleaning:", df.shape)
print("Missing values:", df.isnull().sum().sum())
df.head()


## 3. Handle Outliers

In [ ]:

numeric_cols_without_target = [
    col for col in df.select_dtypes(include=np.number).columns
    if col != "Purchase"
]

for col in numeric_cols_without_target:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower,
              np.where(df[col] > upper, upper, df[col]))

print("Outlier handling completed")


## 4. Separate Features and Target + Encoding

In [ ]:

X = df.drop("Purchase", axis=1)
y = df["Purchase"].astype(int)

print("Target distribution:")
print(y.value_counts())

X = pd.get_dummies(X, drop_first=True)

print("Encoded X shape:", X.shape)


## 5. Feature Selection

In [ ]:

# Memory-efficient feature selection
target_corr = X.corrwith(y).abs().sort_values(ascending=False)

print("Correlation with Purchase:")
print(target_corr)

threshold = 0.005
selected_features = target_corr[target_corr > threshold].index.tolist()

if len(selected_features) == 0:
    X_selected = X
else:
    X_selected = X[selected_features]

print("Selected features:", list(X_selected.columns))
print("Selected X shape:", X_selected.shape)


## 6. Train-Test Split and Scaling

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)


## 7. Default SVM

In [ ]:

default_svm = SVC(kernel="rbf", random_state=42)

default_svm.fit(X_train_scaled, y_train)

y_pred_default = default_svm.predict(X_test_scaled)

default_accuracy = accuracy_score(y_test, y_pred_default)
default_precision = precision_score(y_test, y_pred_default, zero_division=0)
default_recall = recall_score(y_test, y_pred_default, zero_division=0)
default_f1 = f1_score(y_test, y_pred_default, zero_division=0)

print("Default SVM")
print("Accuracy:", default_accuracy)
print("Precision:", default_precision)
print("Recall:", default_recall)
print("F1 Score:", default_f1)


## 8. Hyperparameter Tuning

In [ ]:

# Hyperparameter tuning
# Kept practical because SVM can be slow on 100,000 rows.

param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "class_weight": [None, "balanced"]
}

grid_search = GridSearchCV(
    estimator=SVC(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("Best Cross Validation F1:")
print(grid_search.best_score_)


## 9. Tuned SVM Evaluation

In [ ]:

best_svm = grid_search.best_estimator_

y_pred_tuned = best_svm.predict(X_test_scaled)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
tuned_precision = precision_score(y_test, y_pred_tuned, zero_division=0)
tuned_recall = recall_score(y_test, y_pred_tuned, zero_division=0)
tuned_f1 = f1_score(y_test, y_pred_tuned, zero_division=0)

print("Tuned SVM")
print("Accuracy:", tuned_accuracy)
print("Precision:", tuned_precision)
print("Recall:", tuned_recall)
print("F1 Score:", tuned_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))


## 10. Compare Default and Tuned SVM

In [ ]:

comparison = pd.DataFrame({
    "Model": ["Default SVM", "Tuned SVM"],
    "Accuracy": [default_accuracy, tuned_accuracy],
    "Precision": [default_precision, tuned_precision],
    "Recall": [default_recall, tuned_recall],
    "F1 Score": [default_f1, tuned_f1]
})

display(comparison)

comparison.set_index("Model").plot(kind="bar", figsize=(9, 5))
plt.title("Default vs Tuned SVM Performance")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()


## 11. Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Tuned SVM Confusion Matrix")
plt.show()


## 12. Final Prediction Example

In [ ]:

sample = X_test_scaled[0].reshape(1, -1)
prediction = best_svm.predict(sample)

if prediction[0] == 1:
    print("Customer will purchase")
else:
    print("Customer will not purchase")


## Final Conclusion
The notebook fixes preprocessing errors and tunes SVM hyperparameters using GridSearchCV. Compare the tuned model using accuracy, precision, recall, and F1 score.


# Detailed Conclusion

The Online Purchase Behavior Classification system was successfully developed using a Support Vector Machine (SVM) model to predict whether a customer is likely to purchase a product based on behavioral features such as browsing time, number of clicks, pages visited, add-to-cart count, session duration, product views, and other customer interaction parameters.

The dataset was carefully preprocessed using multiple data handling techniques including duplicate removal, missing value handling, outlier treatment using the IQR method, categorical encoding, feature scaling, and correlation-based feature selection. These preprocessing techniques improved the overall quality and reliability of the dataset and enhanced the model performance.

Hyperparameter tuning using GridSearchCV was applied to optimize the SVM classifier by selecting the best values of parameters such as C, kernel type, gamma, and class weights. The tuned SVM model achieved strong classification performance with high accuracy, precision, recall, and F1 score, making it suitable for large-scale online purchase prediction systems.

The project demonstrates how machine learning can help e-commerce companies analyze customer behavior patterns and predict purchase intent. Such systems can support targeted marketing, product recommendation systems, customer engagement strategies, and business decision-making.

# Future Scope

The future scope of this project is extensive. The system can be improved further by integrating advanced machine learning and deep learning techniques such as Random Forest, XGBoost, Artificial Neural Networks, and transformer-based recommendation systems.

Real-time customer behavior tracking can also be incorporated to perform live purchase prediction while users browse an e-commerce platform. Additional features such as customer demographics, previous purchase history, seasonal trends, payment methods, and product reviews can improve prediction accuracy further.

The project can also be deployed as a real-time web application using frameworks such as Flask or Django. Explainable AI techniques can also be added to improve model interpretability and help businesses understand which customer activities most strongly influence purchase behavior.



# User Input Prediction

This section allows the user to manually enter customer details.  
The tuned SVM model will then predict whether the customer is likely to purchase or not.


In [ ]:

# User Input Prediction using Tuned SVM Model

print("Enter Customer Details:")

user_data = {
    "BrowsingTime(min)": float(input("Browsing Time (min): ")),
    "Clicks": int(input("Number of Clicks: ")),
    "PagesVisited": int(input("Pages Visited: ")),
    "AddToCart": int(input("Add to Cart Count: ")),
    "Category": input("Product Category Viewed: "),
    "WishlistAdd": input("Wishlist Add (Yes/No): "),
    "BounceRate(min)": float(input("Bounce Rate (min): ")),
    "ExitPage": input("Exit Page: "),
    "TimePerPage(min)": float(input("Time Per Page (min): ")),
    "UserType": input("User Type (New/Returning): "),
    "Location": input("Location: "),
    "SearchKeywords": input("Search Keywords: "),
    "SessionDuration(min)": float(input("Session Duration (min): ")),
    "ProductViews": int(input("Product Views: "))
}

# Convert input into dataframe
user_df = pd.DataFrame([user_data])

# Apply same encoding as training data
user_df_encoded = pd.get_dummies(user_df, drop_first=True)

# Match columns with training features
user_df_encoded = user_df_encoded.reindex(columns=X_selected.columns, fill_value=0)

# Scale user input
user_scaled = scaler.transform(user_df_encoded)

# Predict using tuned SVM model
prediction = best_svm.predict(user_scaled)

# Display result
if prediction[0] == 1:
    print("\nPrediction: Customer will purchase")
else:
    print("\nPrediction: Customer will not purchase")
